# Fellegi-Sunter Baseline — Manual Validation Notebook

**Purpose.** Pull 5 random record pairs from each predicted tier (`auto_merge`, `human_review`, `no_match`) produced by `fs_splink_baseline` and manually inspect the match probability against the underlying records.

**Where this runs.** This notebook is authored off-VM but **executes only on the VM** against real `MDM_Population` data. Inputs are auto-resolved to the highest-versioned cleaned parquet + candidate-pairs parquet on disk (same convention as `run_real_baseline.py`).

**Output / artifact.** After **Run All**, the reviewer fills in the *Reviewer judgments* section at the bottom of this notebook (per-pair verdict + notes), saves, commits, pushes. **The committed notebook is the written validation record.**

**PHI note.** Output cells will contain identifier values (names, DOB, SSN, addresses). They stay on the VM. If you commit this notebook with outputs, you are committing PHI to the repo — decide deliberately whether to `Cell → All Output → Clear` before committing.

## 1. Setup & imports

In [ ]:
from __future__ import annotations

import re
import sys
from datetime import datetime
from pathlib import Path

# Project root = two levels up from notebooks/fellegi_sunter/.
PROJECT_ROOT = Path.cwd().resolve().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from models.experiments.fs_splink_baseline import fellegi_sunter_baseline as fs
from src.features.blocking import COL_PATID

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

print(f"PROJECT_ROOT = {PROJECT_ROOT}")

In [ ]:
# Notebook-local constants. Override here if you want a different sample size,
# seed, or to point at alternate input directories.
RANDOM_SEED = 42
SAMPLES_PER_TIER = 5
U_MAX_PAIRS = 1e6  # Matches run_real_baseline.py production default.

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
BLOCKING_DIR = PROJECT_ROOT / "src" / "features" / "outputs" / "blocking"

TIERS = ["auto_merge", "human_review", "no_match"]

## 2. Auto-resolve highest-versioned inputs

Mirrors the version-resolution convention of `src/data/clean.py` and `src/features/run_blocking.py`: parse the integer `v<N>` token from each filename and pick the maximum.

In [ ]:
_VERSION_RE = re.compile(r"_v(\d+)_")


def _latest_versioned(dir_: Path, pattern: str) -> Path:
    """Return the path in `dir_` matching `pattern` with the highest _v<N>_ token."""
    candidates = []
    for p in dir_.glob(pattern):
        m = _VERSION_RE.search(p.name)
        if m:
            candidates.append((int(m.group(1)), p))
    if not candidates:
        raise FileNotFoundError(
            f"No files matching {pattern!r} in {dir_}. "
            "Confirm the cleaning/blocking pipelines have been run on the VM."
        )
    candidates.sort(key=lambda t: t[0])
    return candidates[-1][1]


cleaned_path = _latest_versioned(PROCESSED_DIR, "MDM_Population_cleaned_v*_*.parquet")
pairs_path = _latest_versioned(BLOCKING_DIR, "candidate_pairs_v*_*.parquet")

print(f"Cleaned parquet : {cleaned_path.relative_to(PROJECT_ROOT)}")
print(f"Candidate pairs : {pairs_path.relative_to(PROJECT_ROOT)}")

## 3. Score with `full_output=True`

`full_output=True` returns the classified frame *before* projection to the 5-col evaluation schema, retaining `match_probability`, `match_weight`, `classification_tier`, `source_blocks`, `n_blocks`, and Splink's `gamma_*` per-field agreement levels for downstream inspection.

Training is the slow step (single call to `run_fs_baseline`).

In [ ]:
df_clean = pd.read_parquet(cleaned_path)
print(f"Loaded {len(df_clean):,} cleaned records.")

In [ ]:
df_scored = fs.run_fs_baseline(
    str(pairs_path),
    df_clean,
    u_max_pairs=U_MAX_PAIRS,
    full_output=True,
)

print(f"Scored {len(df_scored):,} candidate pairs.")
print("Tier breakdown:")
print(df_scored["classification_tier"].value_counts().reindex(TIERS, fill_value=0))

## 4. Stratified random sample (5 per tier, fixed seed)

Deterministic across runs given `RANDOM_SEED = 42`. If a tier has fewer than `SAMPLES_PER_TIER` pairs, the notebook takes what's available and prints a warning rather than erroring.

In [ ]:
sampled_chunks = []
for tier in TIERS:
    tier_df = df_scored[df_scored["classification_tier"] == tier]
    n_available = len(tier_df)
    n_take = min(SAMPLES_PER_TIER, n_available)
    if n_take < SAMPLES_PER_TIER:
        print(
            f"WARN: tier {tier!r} has only {n_available} pairs; sampling {n_take}."
        )
    sampled_chunks.append(tier_df.sample(n=n_take, random_state=RANDOM_SEED))

sampled = pd.concat(sampled_chunks).reset_index(drop=True)
print(f"Sampled {len(sampled)} pairs total.")
print(sampled["classification_tier"].value_counts().reindex(TIERS, fill_value=0))

## 5. Side-by-side identifier display

For each sampled pair the notebook renders:

1. A one-line header with `classification_tier`, `match_probability`, `match_weight`, and which blocks fired.
2. A two-column DataFrame (`Record A` vs `Record B`) of the cleaned identifier fields. Values are pulled from the cleaned dataframe by `PATID` — this avoids depending on Splink's `_l/_r` suffix convention and shows all fields whether or not the model used them as evidence.
3. A compact view of Splink's `gamma_*` per-field agreement levels (when present), so you can see *why* the model assigned the score it did.

In [ ]:
# Fields to display per record, in order. (Label, cleaned-dataframe column name.)
DISPLAY_FIELDS: list[tuple[str, str]] = [
    ("PATID",              "PATID"),
    ("First name",         "FirstNM_clean"),
    ("Middle name",        "MiddleNM_clean"),
    ("Last name",          "LastNM_clean"),
    ("Full name tokens",   "full_name_tokens"),
    ("DOB",                "BirthDT_clean"),
    ("SSN (full)",         "SSN_clean"),
    ("SSN last-4",         "last_4_SSN"),
    ("Email",              "Email_clean"),
    ("Address line 1",     "AddressLine1_clean"),
    ("Address line 2",     "AddressLine2_clean"),
    ("City",               "CityNM_clean"),
    ("State",              "StateCD_clean"),
    ("ZIP",                "ZipCD_clean_base"),
    ("Phones (set)",       "Phones_set"),
]

# Index df_clean by PATID once for O(1) lookups.
_clean_indexed = df_clean.set_index(COL_PATID, drop=False)


def _lookup(patid: str) -> pd.Series:
    """Pull one cleaned record by PATID; returns an empty Series if absent."""
    try:
        return _clean_indexed.loc[patid]
    except KeyError:
        return pd.Series(dtype=object)


def render_identifier_table(patid_a: str, patid_b: str) -> pd.DataFrame:
    """Two-column side-by-side DataFrame of cleaned identifier fields."""
    rec_a = _lookup(patid_a)
    rec_b = _lookup(patid_b)
    rows = {}
    for label, col in DISPLAY_FIELDS:
        if col not in df_clean.columns:
            continue  # field absent in this cleaned parquet version; skip.
        rows[label] = [rec_a.get(col, np.nan), rec_b.get(col, np.nan)]
    return pd.DataFrame.from_dict(
        rows, orient="index", columns=["Record A", "Record B"]
    )


def render_gamma_table(row: pd.Series) -> pd.DataFrame | None:
    """Splink's gamma_<field> per-field agreement levels, if retained."""
    gamma_cols = [c for c in row.index if c.startswith("gamma_")]
    if not gamma_cols:
        return None
    out = pd.DataFrame(
        {"agreement_level": [row[c] for c in gamma_cols]},
        index=[c.removeprefix("gamma_") for c in gamma_cols],
    )
    out.index.name = "comparison"
    return out

In [ ]:
from IPython.display import display, Markdown

for i, row in sampled.iterrows():
    pair_num = i + 1
    patid_a = row["PATID_A"]
    patid_b = row["PATID_B"]
    tier = row["classification_tier"]
    p = row.get("match_probability", float("nan"))
    w = row.get("match_weight", float("nan"))
    src_blocks = row.get("source_blocks", "")
    n_blocks = row.get("n_blocks", "")

    display(Markdown(
        f"### Pair {pair_num}/{len(sampled)} \u2014 tier=`{tier}`  \n"
        f"`PATID_A={patid_a}` \u2194 `PATID_B={patid_b}`  \n"
        f"`match_probability={p:.4f}`  \u00b7  `match_weight={w:.3f}`  "
        f"\u00b7  `n_blocks={n_blocks}`  \u00b7  `source_blocks={src_blocks}`"
    ))
    display(render_identifier_table(patid_a, patid_b))
    gammas = render_gamma_table(row)
    if gammas is not None:
        display(Markdown("**Splink agreement levels (`gamma_*`):**"))
        display(gammas)

## 6. Reviewer judgment templates

Run the cell below to generate one markdown template per sampled pair. Copy the printed block into the **Reviewer judgments** markdown cell at the bottom of this notebook and fill in `Reviewer verdict` (`true_match` / `not_match` / `unsure`) and `Reviewer notes` for each pair. Save, commit, push.

Why a printed block (not interactive widgets): markdown survives `nbconvert`, diffs cleanly in git, and the committed notebook is then a self-contained validation record.

In [ ]:
template_lines = []
for i, row in sampled.iterrows():
    pair_num = i + 1
    template_lines.append(
        f"### Pair {pair_num}/{len(sampled)} \u2014 "
        f"PATID_A={row['PATID_A']}, PATID_B={row['PATID_B']}\n"
        f"\n"
        f"- **Predicted tier:** `{row['classification_tier']}`\n"
        f"- **Model match_probability:** {row.get('match_probability', float('nan')):.4f}\n"
        f"- **Reviewer verdict:** [ true_match | not_match | unsure ]\n"
        f"- **Reviewer notes:**\n"
        f"  - \n"
    )

print("\n".join(template_lines))

## 7. Diagnostic summary (non-PHI)

Provenance trail recording which dataset version was validated. Safe to commit even when output cells are cleared.

In [ ]:
tier_counts = df_scored["classification_tier"].value_counts().reindex(TIERS, fill_value=0)
total = int(tier_counts.sum())

print(f"Executed at         : {datetime.now().isoformat(timespec='seconds')}")
print(f"Cleaned parquet     : {cleaned_path.relative_to(PROJECT_ROOT)}")
print(f"Candidate pairs     : {pairs_path.relative_to(PROJECT_ROOT)}")
print(f"Cleaned records     : {len(df_clean):,}")
print(f"Candidate pairs     : {len(df_scored):,}")
print(f"Random seed         : {RANDOM_SEED}")
print(f"Samples per tier    : {SAMPLES_PER_TIER}")
print()
print("Tier distribution (full population):")
for tier in TIERS:
    n = int(tier_counts[tier])
    pct = (n / total * 100) if total else 0.0
    print(f"  {tier:<14s} {n:>10,}  ({pct:5.2f}%)")

## Reviewer judgments

_Paste the templates printed in Section 6 below this line, then fill in each `Reviewer verdict` and `Reviewer notes`._

<!-- BEGIN JUDGMENTS -->

<!-- END JUDGMENTS -->